# GPT timing attack

We attack possible content-moderation gate surface. Same attack could be used for Cash-timing attacks or can exploit shared endpoint output class.

In [ ]:
!pip install torch scikit-learn numpy --break-system-packages -q 2>&1 | tail -5

In [ ]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass
from sklearn.linear_model import LogisticRegression

torch.manual_seed(1)
rng = np.random.default_rng(1)

## Model

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout

        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True
            )
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))
        print("number of parameters: %.2fM" % (self.get_num_params() / 1e6,))

    def get_num_params(self, non_embedding=True):
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)


In [ ]:
CONFIG_MODE = "full"   # "small" for chack, "full" for 124M GPT-2

if CONFIG_MODE == "small":
    cfg = GPTConfig(block_size=16, vocab_size=100, n_layer=6, n_head=4, n_embd=32, bias=True)
    ADD_SYNTHETIC_DELAY = True   # On CPU, the difference between 2 and 6 micro-layers is lost in the noise without amplification.
else:
    cfg = GPTConfig()
    cfg.block_size = 256           # context length
    ADD_SYNTHETIC_DELAY = False  # diff is fine

model = GPT(cfg)
model.eval()
print(cfg)


number of parameters: 123.69M
GPTConfig(block_size=256, vocab_size=50304, n_layer=12, n_head=12, n_embd=768, dropout=0.0, bias=True)


The exit_gate is the secret that the attacker is trying to extract. It is positioned after EXIT_AFTER_LAYER blocks. The attacker directly controls the embedding of the last sequence position, context is fixed and its content is unknown to the attacker. The attacker does not observe logits or hidden states—only the response latency.

In [ ]:
EXIT_AFTER_LAYER = 2
exit_gate = nn.Linear(cfg.n_embd, 1)          # Secret
n_exit_params = cfg.n_embd + 1
print(f"Params in exit_gate: {n_exit_params}")

T = 8
context_ids = torch.randint(0, cfg.vocab_size, (1, T - 1))  # context


@torch.no_grad()
def build_sequence_embedding(x_last_np):
    ctx_emb = model.transformer.wte(context_ids) + model.transformer.wpe(torch.arange(T - 1).unsqueeze(0))
    pos_last = torch.tensor([[T - 1]])
    x_last = torch.tensor(x_last_np, dtype=torch.float32).view(1, 1, -1)
    last_emb = x_last + model.transformer.wpe(pos_last)
    return torch.cat([ctx_emb, last_emb], dim=1)


@torch.no_grad()
def run_blocks(x, start, end):
    for blk in model.transformer.h[start:end]:
        x = blk(x)
    return x


@torch.no_grad()
def enclave_forward(x_last_np, simulate_timing=False):
    x = build_sequence_embedding(x_last_np)
    t0 = time.perf_counter() if simulate_timing else None

    x = run_blocks(x, 0, EXIT_AFTER_LAYER)
    h_last = x[0, -1, :]
    will_exit = (exit_gate(h_last) > 0).item()

    if not will_exit:
        x = run_blocks(x, EXIT_AFTER_LAYER, cfg.n_layer)

    n_layers_run = EXIT_AFTER_LAYER if will_exit else cfg.n_layer
    latency = None
    if simulate_timing:
        latency = time.perf_counter() - t0
        if ADD_SYNTHETIC_DELAY:
            latency += n_layers_run * 2e-4
    return will_exit, latency


@torch.no_grad()
def true_exit_decision(x_last_np):
    x = build_sequence_embedding(x_last_np)
    x = run_blocks(x, 0, EXIT_AFTER_LAYER)
    return int((exit_gate(x[0, -1, :]) > 0).item())


Params in exit_gate: 769


## Attacker oracle

In [ ]:
def timing_oracle(x_last_np, reps=1):
    lat = [enclave_forward(x_last_np, simulate_timing=True)[1] for _ in range(reps)]
    return float(np.median(lat))


def calibrate_threshold():
    lo = timing_oracle(np.full(cfg.n_embd, -6.0), reps=5)
    hi = timing_oracle(np.full(cfg.n_embd, 6.0), reps=5)
    return (lo + hi) / 2


t_thresh = calibrate_threshold()
print(f"timing threshold: {t_thresh*1000:.2f} ms")


def fast_path_label(x_last_np):
    return 1 if timing_oracle(x_last_np) < t_thresh else 0

timing threshold: 65.91 ms


## Evals

In [ ]:
QUERY_BUDGET_MULTIPLIER = 5     # ~5x gate
n_rounds = 15
total_budget = QUERY_BUDGET_MULTIPLIER * n_exit_params
queries_per_round = max(10, total_budget // n_rounds)

X_all, y_all = [], []
surrogate = None
n_queries_total = 0
t_start = time.perf_counter()

for round_idx in range(n_rounds):
    if surrogate is None:
        batch = rng.normal(scale=3.0, size=(queries_per_round, cfg.n_embd))
    else:
        w_s, b_s = surrogate.coef_[0], surrogate.intercept_[0]
        batch = []
        for _ in range(queries_per_round):
            x0 = rng.normal(scale=3.0, size=cfg.n_embd)
            step = -(w_s * (w_s @ x0 + b_s)) / (w_s @ w_s)
            x_b = x0 + step + rng.normal(scale=0.5, size=cfg.n_embd)
            batch.append(x_b)
        batch = np.array(batch)

    for x in batch:
        y = fast_path_label(x)
        n_queries_total += 1
        X_all.append(x); y_all.append(y)

    X_arr, y_arr = np.array(X_all), np.array(y_all)
    if len(set(y_arr)) < 2:
        continue
    surrogate = LogisticRegression(max_iter=2000).fit(X_arr, y_arr)
    if round_idx % 3 == 0 or round_idx == n_rounds - 1:
        print(f"  round {round_idx}: {len(X_arr)} dots, train acc={surrogate.score(X_arr, y_arr):.3f}, "
              f"elapsed={time.perf_counter()-t_start:.1f}s")

print(f"\n Calls: {n_queries_total} (~{n_queries_total/n_exit_params:.1f}x params gate)")


  раунд 0: 256 точек, train acc=1.000, elapsed=25.7s
  раунд 3: 1024 точек, train acc=1.000, elapsed=63.1s
  раунд 6: 1792 точек, train acc=1.000, elapsed=98.5s
  раунд 9: 2560 точек, train acc=1.000, elapsed=134.2s
  раунд 12: 3328 точек, train acc=0.999, elapsed=171.1s
  раунд 14: 3840 точек, train acc=0.999, elapsed=193.9s

 Calls: 3840 (~5.0x params gate)


In [ ]:
n_test = 300
X_test = rng.normal(scale=3.0, size=(n_test, cfg.n_embd))
y_true = np.array([true_exit_decision(x) for x in X_test])
y_pred_from_timing = np.array([fast_path_label(x) for x in X_test])
y_pred_surrogate = surrogate.predict(X_test)

print(f"Oracle alignment:"
      f"{(y_true == y_pred_from_timing).mean()*100:.1f}%")
print(f"Stolen alignment:"
      f"{(y_true == y_pred_surrogate).mean()*100:.1f}%")


Oracle alignment:100.0%
Stolen alignment:93.7%


# Semantic attack